[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/45.%20The%20Terminal%20Digital%20Twin%20Scoping%20Problem/P45-Tier-5_executed.ipynb)

---


# 45. The Terminal Digital Twin Scoping Problem
## Tier 5: The Integrated Digital Twin (System-of-Systems Simulation)

### Goal
Build a small, rule-based system-of-systems simulation for a container terminal. The example represents five subsystem twins—Vessel, Yard, Gate, Rail, and Equipment—and a coordinator that runs their scenario calculations in sequence and exchanges status messages. The purpose is to make subsystem dependencies and scenario reasoning visible; this notebook does not implement a calibrated operational digital twin or an optimizer.

### Inputs and outputs
The Black Friday scenario uses the stated demand increase, duration, and initial terminal state as inputs. Each subsystem applies its own explicit rules to those inputs. The outputs are rule-based forecasts and indicators, a record of subsystem messages, a visualization of the modeled quantities, and a separate vessel-delay what-if analysis.

### Background: states, twins, and coordination
A **state** is a collection of values describing a system at a particular time. Here, `TerminalState` stores values such as queue lengths, yard utilization, equipment availability, and throughput. A **subsystem twin** is a software representation of one operational area; it stores its own history and can send or process messages. The **coordinator** registers the twins, establishes their communication links, and runs subsystem calculations in a defined order.

A communication link makes message exchange possible; it does not, by itself, make an optimization coordinated. In this example, the coordinator sends status messages and processes them, while each subsystem's numerical rule remains the one shown in its own method. The example therefore demonstrates an integrated architecture and a deterministic scenario workflow, not a feedback-controlled or empirically validated simulation.

### Model assumptions and scope
- The scenario is deterministic: the same inputs produce the same rule-based outputs, apart from timestamps used for forecasts and messages.
- Subsystem rules are illustrative relationships, not estimates fitted to terminal operating data. Parameters and constants are model assumptions.
- A message communicates an update; it does not automatically change another subsystem's calculations unless a handler explicitly uses it.
- The Black Friday case values and delay effects are scenario inputs. The model does not establish that any reported outcome has occurred at a real terminal.
- The model omits detailed schedules, finite-capacity constraints, stochastic arrivals, feedback control, calibration, and a common monetary valuation of all performance measures.
- Consequently, results should be interpreted as outputs of the stated rules, not as evidence of realized savings, causal coordination benefits, or an optimal operating plan.

### Modeling logic
The scenario calculation is a sequence of explicit subsystem rules. For example, the vessel departure forecast is computed as

$$
T_{\mathrm{forecast}} = T_{\mathrm{base}} + 2Q_v + 12(1-A_e),
$$

where $T_{\mathrm{base}}$ is the assumed base processing time in hours, $Q_v$ is the vessel queue length, and $A_e$ is equipment availability expressed as a fraction between 0 and 1. The code implements these terms as `base_processing_time`, `queue_factor`, and `equipment_factor`. This is a forecast rule, not an optimization objective.

The coordinator's sequence is

$$
\text{initial state}
\rightarrow \text{vessel forecast}
\rightarrow \text{yard rule}
\rightarrow \text{gate rule}
\rightarrow \text{rail rule}
\rightarrow \text{equipment rule}
\rightarrow \text{reported indicators}.
$$

Each step uses the inputs shown in its method. The model has no Bellman equation, search procedure, or solver-based objective: it evaluates a fixed set of deterministic rules. With five subsystem calculations and a fixed number of messages, scenario work is constant in the number of scenario inputs; sorting a message queue containing $m$ messages takes $O(m\log m)$ time. The stored state history is capped at 100 entries per twin.

### Approach
1. Define the terminal state, message format, and common subsystem-twin behavior.
2. Implement one rule-based twin for each of the five operational areas.
3. Register the twins and establish a fully connected communication network.
4. Run the Black Friday scenario in the coordinator's fixed sequence.
5. Inspect the rule outputs and visualize quantities with their units.
6. Evaluate a separate vessel-delay scenario to illustrate assumed cascading effects.

### Illustrative case context
The motivating case description gives a five-day peak-period scenario with a 35% volume increase and mentions possible targets such as more vessel calls, higher yard capacity, shorter gate processing time, greater rail share, and improved equipment availability. Those figures are case-context claims, not outputs validated by this notebook. The code only reports values calculated from its stated rules and inputs.

In [ ]:
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass
from datetime import datetime, timedelta

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("default")
sns.set_palette("husl")

print("Imports and plotting style are ready.")

In [ ]:
@dataclass
class TerminalState:
    """Snapshot of the terminal quantities used by the scenario rules."""

    timestamp: datetime
    vessel_berths_occupied: int
    vessel_queue_length: int
    yard_utilization: float
    yard_container_count: int
    gate_queue_length: int
    gate_processing_time_avg: float
    rail_slots_available: int
    rail_queue_length: int
    equipment_availability: float
    equipment_maintenance_scheduled: int
    total_containers_moved: int
    system_throughput: float


@dataclass
class SubSystemMessage:
    """Status message passed from one subsystem twin to another."""

    sender: str
    receiver: str
    message_type: str
    data: Dict
    timestamp: datetime
    priority: int = 2  # 1 is highest priority; larger numbers are lower priority.


class SubSystemTwin:
    """Shared message and state-history behavior for a subsystem twin."""

    def __init__(self, name: str, twin_id: str):
        self.name = name
        self.twin_id = twin_id
        self.state_history: List[TerminalState] = []
        self.message_queue: List[SubSystemMessage] = []
        self.connected_twins: List["SubSystemTwin"] = []
        self.performance_metrics = {
            "throughput": [],
            "utilization": [],
            "efficiency": [],
            "response_time": [],
            "coordination_benefits": [],
        }

    def connect_to(self, other_twin: "SubSystemTwin") -> None:
        """Record a bidirectional link so either twin can exchange messages."""
        if other_twin is self:
            return
        if other_twin not in self.connected_twins:
            self.connected_twins.append(other_twin)
        if self not in other_twin.connected_twins:
            other_twin.connected_twins.append(self)

    def send_message(
        self,
        receiver: str,
        message_type: str,
        data: Dict,
        priority: int = 2,
    ) -> SubSystemMessage:
        """Create a timestamped message; delivery is handled by the coordinator."""
        return SubSystemMessage(
            sender=self.twin_id,
            receiver=receiver,
            message_type=message_type,
            data=data,
            timestamp=datetime.now(),
            priority=priority,
        )

    def receive_message(self, message: SubSystemMessage) -> None:
        """Queue a message so higher-priority updates are processed first."""
        self.message_queue.append(message)
        self.message_queue.sort(key=lambda queued_message: queued_message.priority)

    def process_messages(self) -> None:
        """Process every queued update in priority order."""
        while self.message_queue:
            message = self.message_queue.pop(0)
            self._handle_message(message)

    def _handle_message(self, message: SubSystemMessage) -> None:
        """Subclass hook for message types understood by a particular twin."""
        del message

    def update_state(self, terminal_state: TerminalState) -> None:
        """Store a snapshot while bounding memory use to recent history."""
        self.state_history.append(terminal_state)
        if len(self.state_history) > 100:
            self.state_history.pop(0)


print("Terminal state, messages, and shared twin behavior are defined.")

In [ ]:
class VesselOperationsTwin(SubSystemTwin):
    """Rule-based representation of vessel operations."""

    def __init__(self):
        super().__init__("Vessel Operations", "VT")
        self.berth_schedule = {}
        self.crane_assignments = {}
        self.vessel_predictions = {}
        self.berth_utilization = []
        self.coordination_updates = []

    def _handle_message(self, message: SubSystemMessage) -> None:
        """Retain received status updates for inspection by vessel operations."""
        if message.message_type in {
            "YARD_UPDATE",
            "GATE_CONGESTION",
            "EQUIPMENT_STATUS",
        }:
            self.coordination_updates.append(
                {
                    "message_type": message.message_type,
                    "sender": message.sender,
                    "data": message.data,
                }
            )

    def predict_vessel_departure_times(
        self, terminal_state: TerminalState
    ) -> Dict:
        """Apply a transparent queue-and-equipment forecast rule."""
        base_processing_time = 24  # Assumed base time, in hours.
        queue_factor = terminal_state.vessel_queue_length * 2
        equipment_factor = (1.0 - terminal_state.equipment_availability) * 12
        predicted_time = base_processing_time + queue_factor + equipment_factor

        predictions = {
            "next_vessel": {
                "estimated_departure": datetime.now()
                + timedelta(hours=predicted_time),
                "forecast_horizon_hours": predicted_time,
                "confidence": 0.85,
                "factors": {
                    "base_processing_time_hours": base_processing_time,
                    "queue_impact_hours": queue_factor,
                    "equipment_impact_hours": equipment_factor,
                },
            }
        }
        self.vessel_predictions = predictions
        return predictions


print("Vessel Operations Twin is ready.")

In [ ]:
class YardManagementTwin(SubSystemTwin):
    """Rule-based representation of yard capacity responses."""

    def __init__(self):
        super().__init__("Yard Management", "YT")
        self.container_locations = {}
        self.block_utilization = {}
        self.stack_heights = {}
        self.repositioning_plan = {}
        self.yard_capacity_expansion = 0

    def optimize_yard_capacity(self, demand_increase: float) -> Dict:
        """Estimate capacity response using the stated strategy fractions."""
        if demand_increase < 0:
            raise ValueError("demand_increase must be non-negative")

        optimization_strategies = {
            "stack_height_increase": min(demand_increase * 0.3, 0.5),
            "empty_container_relocation": demand_increase * 0.2,
            "block_reallocation": demand_increase * 0.1,
            "temporary_storage": demand_increase * 0.15,
        }
        capacity_increase = sum(optimization_strategies.values())

        # The cost rule is stated per unit fractional capacity increase.
        return {
            "strategies": optimization_strategies,
            "capacity_increase_fraction": capacity_increase,
            "capacity_increase_percent": capacity_increase * 100,
            "estimated_cost": capacity_increase * 1000,
            "estimated_cost_units": "model cost units",
            "implementation_time_hours": capacity_increase * 24,
        }


print("Yard Management Twin is ready.")

In [ ]:
class GateOperationsTwin(SubSystemTwin):
    """Rule-based representation of gate appointments and truck flow."""

    def __init__(self):
        super().__init__("Gate Operations", "GT")
        self.truck_appointments = {}
        self.gate_lanes_status = {}
        self.processing_times = {}
        self.queue_predictions = {}
        self.appointment_optimization = {}

    def optimize_truck_flow(self, container_availability_data: Dict) -> Dict:
        """Redistribute appointments to slots with high container availability."""
        availability_patterns = container_availability_data.get("patterns", {})
        appointment_redistribution = {}

        for time_slot, availability in availability_patterns.items():
            if availability > 0.8:
                appointment_redistribution[time_slot] = {
                    "increase_capacity_fraction": 0.2,
                    "target_utilization": 0.9,
                }

        total_improvement = len(appointment_redistribution) * 0.05
        return {
            "appointment_redistribution": appointment_redistribution,
            "lane_allocation_optimization": {},
            "processing_time_improvement": min(total_improvement, 0.25),
        }


print("Gate Operations Twin is ready.")

In [ ]:
class RailOperationsTwin(SubSystemTwin):
    """Rule-based representation of rail capacity negotiation."""

    def __init__(self):
        super().__init__("Rail Operations", "RT")
        self.train_schedule = {}
        self.track_utilization = {}
        self.rail_capacity_negotiation = {}
        self.container_transfer_plan = {}

    def negotiate_additional_capacity(self, demand_increase: float) -> Dict:
        """Estimate additional rail share using the stated capacity rule."""
        if demand_increase < 0:
            raise ValueError("demand_increase must be non-negative")

        current_capacity = 0.15
        target_capacity = min(current_capacity + demand_increase * 0.05, 0.25)
        additional_capacity = target_capacity - current_capacity

        return {
            "current_capacity_share": current_capacity,
            "target_capacity_share": target_capacity,
            "additional_capacity_secured": additional_capacity,
            "additional_capacity_percentage_points": additional_capacity * 100,
            "additional_trains_per_day": int(additional_capacity * 10),
            "negotiated_cost_per_container": 25,
            "implementation_time_hours": 48,
        }


class EquipmentMaintenanceTwin(SubSystemTwin):
    """Rule-based representation of equipment availability responses."""

    def __init__(self):
        super().__init__("Equipment Maintenance", "ET")
        self.equipment_health = {}
        self.maintenance_schedule = {}
        self.predictive_maintenance = {}
        self.availability_optimization = {}

    def optimize_availability_for_surge(
        self, surge_duration_hours: int
    ) -> Dict:
        """Apply duration thresholds and cap the resulting availability."""
        if surge_duration_hours < 0:
            raise ValueError("surge_duration_hours must be non-negative")

        optimization_strategies = {
            "defer_noncritical_maintenance": surge_duration_hours > 24,
            "increase_inspection_frequency": surge_duration_hours > 48,
            "standby_equipment_activation": surge_duration_hours > 72,
            "extended_shift_coverage": surge_duration_hours > 12,
        }

        base_availability = 0.94
        availability_improvement = 0.0
        if optimization_strategies["defer_noncritical_maintenance"]:
            availability_improvement += 0.02
        if optimization_strategies["standby_equipment_activation"]:
            availability_improvement += 0.015
        if optimization_strategies["extended_shift_coverage"]:
            availability_improvement += 0.01

        optimized_availability = min(
            base_availability + availability_improvement, 0.98
        )
        actual_improvement = optimized_availability - base_availability

        return {
            "strategies": optimization_strategies,
            "base_availability": base_availability,
            "optimized_availability": optimized_availability,
            "availability_improvement": actual_improvement,
            "estimated_cost_savings": availability_improvement * 50000,
        }


print("Rail Operations and Equipment Maintenance Twins are ready.")

In [ ]:
class IntegratedDigitalTwinCoordinator:
    """Register twins, deliver messages, and run the fixed scenario sequence."""

    def __init__(self):
        self.subsystem_twins: Dict[str, SubSystemTwin] = {}
        self.system_state_history: List[TerminalState] = []
        self.coordination_metrics = {
            "message_flow": [],
            "response_times": [],
            "coordination_benefits": [],
            "system_efficiency": [],
            "emergent_behaviors": [],
        }
        self.scenario_results = {}

    def register_subsystem_twin(self, twin: SubSystemTwin) -> None:
        """Make a uniquely identified twin available to the coordinator."""
        if twin.twin_id in self.subsystem_twins:
            raise ValueError(f"Twin ID {twin.twin_id!r} is already registered")
        self.subsystem_twins[twin.twin_id] = twin

    def establish_connections(self) -> None:
        """Create a fully connected network among registered subsystem twins."""
        twin_list = list(self.subsystem_twins.values())
        for index, twin1 in enumerate(twin_list):
            for twin2 in twin_list[index + 1 :]:
                twin1.connect_to(twin2)

    def _deliver_message(
        self,
        sender_id: str,
        receiver_id: str,
        message_type: str,
        data: Dict,
        priority: int = 2,
    ) -> None:
        """Route one update over an established twin connection."""
        sender = self.subsystem_twins[sender_id]
        receiver = self.subsystem_twins[receiver_id]
        if receiver not in sender.connected_twins:
            raise RuntimeError("A message cannot use a connection that is not established")

        message = sender.send_message(
            receiver_id, message_type, data, priority=priority
        )
        receiver.receive_message(message)
        self.coordination_metrics["message_flow"].append(message)

    def simulate_black_friday_scenario(self) -> Dict:
        """Run the stated peak-demand scenario through the subsystem rules."""
        required_ids = {"VT", "YT", "GT", "RT", "ET"}
        if not required_ids.issubset(self.subsystem_twins):
            raise RuntimeError("Register all five subsystem twins before simulation")

        print("=== BLACK FRIDAY PEAK-SEASON SCENARIO ===")
        demand_increase = 0.35
        duration_days = 5
        additional_vessels = 12

        initial_state = TerminalState(
            timestamp=datetime.now(),
            vessel_berths_occupied=6,
            vessel_queue_length=2,
            yard_utilization=0.75,
            yard_container_count=12000,
            gate_queue_length=25,
            gate_processing_time_avg=45,
            rail_slots_available=8,
            rail_queue_length=3,
            equipment_availability=0.94,
            equipment_maintenance_scheduled=5,
            total_containers_moved=800,
            system_throughput=0.85,
        )
        self.system_state_history.append(initial_state)

        simulation_results = self._run_coordinated_simulation(
            initial_state, demand_increase, duration_days, additional_vessels
        )
        self.scenario_results[simulation_results["scenario"]] = simulation_results
        return simulation_results

    def _run_coordinated_simulation(
        self,
        initial_state: TerminalState,
        demand_increase: float,
        duration_days: int,
        additional_vessels: int,
    ) -> Dict:
        """Evaluate subsystem rules in sequence and exchange status updates."""
        print("\nPhase 1: Vessel operations forecast")
        vessel_twin = self.subsystem_twins["VT"]
        vessel_predictions = vessel_twin.predict_vessel_departure_times(initial_state)

        print("Phase 2: Yard capacity response")
        yard_twin = self.subsystem_twins["YT"]
        yard_optimization = yard_twin.optimize_yard_capacity(demand_increase)
        self._deliver_message(
            "YT",
            "VT",
            "YARD_UPDATE",
            {
                "capacity_increase_percent": yard_optimization[
                    "capacity_increase_percent"
                ]
            },
        )

        print("Phase 3: Gate flow response")
        gate_twin = self.subsystem_twins["GT"]
        container_availability = {
            "patterns": {"morning": 0.85, "afternoon": 0.90, "evening": 0.80}
        }
        gate_optimization = gate_twin.optimize_truck_flow(
            container_availability
        )
        self._deliver_message(
            "GT",
            "VT",
            "GATE_CONGESTION",
            {
                "processing_time_improvement_fraction": gate_optimization[
                    "processing_time_improvement"
                ]
            },
        )

        print("Phase 4: Rail capacity response")
        rail_twin = self.subsystem_twins["RT"]
        rail_negotiation = rail_twin.negotiate_additional_capacity(demand_increase)

        print("Phase 5: Equipment availability response")
        equipment_twin = self.subsystem_twins["ET"]
        equipment_optimization = equipment_twin.optimize_availability_for_surge(
            duration_days * 24
        )
        self._deliver_message(
            "ET",
            "VT",
            "EQUIPMENT_STATUS",
            {
                "optimized_availability": equipment_optimization[
                    "optimized_availability"
                ]
            },
        )

        # Handlers retain the messages for inspection; the fixed rules above are
        # not recalculated from those messages, so no feedback effect is implied.
        for twin in self.subsystem_twins.values():
            twin.process_messages()

        return {
            "scenario": "Black Friday Peak Season",
            "demand_increase": demand_increase,
            "duration_days": duration_days,
            "additional_vessels": additional_vessels,
            "initial_state": initial_state,
            "subsystem_optimizations": {
                "vessel_operations": vessel_predictions,
                "yard_management": yard_optimization,
                "gate_operations": gate_optimization,
                "rail_operations": rail_negotiation,
                "equipment_maintenance": equipment_optimization,
            },
            "messages_delivered": len(self.coordination_metrics["message_flow"]),
        }


print("Integrated Digital Twin Coordinator is ready.")

In [ ]:
# Create one twin for each modeled operational area.
vessel_twin = VesselOperationsTwin()
yard_twin = YardManagementTwin()
gate_twin = GateOperationsTwin()
rail_twin = RailOperationsTwin()
equipment_twin = EquipmentMaintenanceTwin()

# The coordinator owns registration and the communication network.
coordinator = IntegratedDigitalTwinCoordinator()
for twin in [
    vessel_twin,
    yard_twin,
    gate_twin,
    rail_twin,
    equipment_twin,
]:
    coordinator.register_subsystem_twin(twin)

coordinator.establish_connections()

print("=== INTEGRATED DIGITAL TWIN SYSTEM INITIALIZED ===")
print(f"Registered subsystem twins: {len(coordinator.subsystem_twins)}")
for twin in coordinator.subsystem_twins.values():
    connected_names = [connected_twin.name for connected_twin in twin.connected_twins]
    print(f"{twin.name}: connected to {len(connected_names)} other twins")
    for connected_name in connected_names:
        print(f"  - {connected_name}")

assert all(
    len(twin.connected_twins) == len(coordinator.subsystem_twins) - 1
    for twin in coordinator.subsystem_twins.values()
)

In [ ]:
simulation_results = coordinator.simulate_black_friday_scenario()
subsystem_results = simulation_results["subsystem_optimizations"]

print("\n=== RULE-BASED SCENARIO OUTPUTS ===")
print(f"Scenario: {simulation_results['scenario']}")
print(f"Demand increase input: {simulation_results['demand_increase']:.1%}")
print(f"Duration input: {simulation_results['duration_days']} days")
print(f"Additional vessel calls in scenario description: {simulation_results['additional_vessels']}")
print(f"Status messages delivered: {simulation_results['messages_delivered']}")

vessel_result = subsystem_results["vessel_operations"]["next_vessel"]
print("\nVessel Operations:")
print(
    "  Forecast horizon: "
    f"{vessel_result['forecast_horizon_hours']:.2f} hours"
)
print(f"  Forecast confidence assumption: {vessel_result['confidence']:.1%}")
for factor, value in vessel_result["factors"].items():
    print(f"  {factor.replace('_', ' ').title()}: {value:.2f} hours")

yard_result = subsystem_results["yard_management"]
print("\nYard Management:")
print(f"  Rule-based capacity increase: {yard_result['capacity_increase_percent']:.2f}%")
print(
    "  Estimated implementation cost: "
    f"{yard_result['estimated_cost']:.2f} {yard_result['estimated_cost_units']}"
)
print(
    "  Estimated implementation time: "
    f"{yard_result['implementation_time_hours']:.2f} hours"
)

gate_result = subsystem_results["gate_operations"]
print("\nGate Operations:")
print(
    "  Rule-based processing-time improvement: "
    f"{gate_result['processing_time_improvement']:.1%}"
)
print(
    "  Appointment slots redistributed: "
    f"{len(gate_result['appointment_redistribution'])}"
)

rail_result = subsystem_results["rail_operations"]
print("\nRail Operations:")
print(
    "  Additional capacity share: "
    f"{rail_result['additional_capacity_percentage_points']:.2f} percentage points"
)
print(f"  Additional trains per day under the rule: {rail_result['additional_trains_per_day']}")
print(
    "  Negotiated cost assumption: "
    f"${rail_result['negotiated_cost_per_container']} per container"
)

equipment_result = subsystem_results["equipment_maintenance"]
print("\nEquipment Maintenance:")
print(
    "  Base availability: "
    f"{equipment_result['base_availability']:.1%}"
)
print(
    "  Rule-based optimized availability: "
    f"{equipment_result['optimized_availability']:.1%}"
)
print(
    "  Availability change: "
    f"{equipment_result['availability_improvement'] * 100:.2f} percentage points"
)
print(
    "  Estimated savings under the stated rule: "
    f"${equipment_result['estimated_cost_savings']:,.2f}"
)

print("\nCoordination messages retained by Vessel Operations:")
for update in vessel_twin.coordination_updates:
    print(f"  {update['message_type']} from {update['sender']}: {update['data']}")

assert simulation_results["messages_delivered"] == 3
assert len(vessel_twin.coordination_updates) == 3
assert 0 <= equipment_result["optimized_availability"] <= 1

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle(
    "Rule-Based Integrated Digital Twin Scenario",
    fontsize=16,
    fontweight="bold",
)

subsystem_results = simulation_results["subsystem_optimizations"]
vessel_factors = subsystem_results["vessel_operations"]["next_vessel"]["factors"]

# Forecast components are shown in hours so the total can be interpreted directly.
ax = axes[0, 0]
factor_names = [name.replace("_hours", "").replace("_", " ").title() for name in vessel_factors]
factor_values = list(vessel_factors.values())
ax.bar(factor_names, factor_values, color="#4C78A8", edgecolor="black")
ax.set_ylabel("Forecast contribution (hours)")
ax.set_title("Vessel Forecast Components")
ax.tick_params(axis="x", rotation=25)
ax.grid(True, alpha=0.3, axis="y")

yard_strategies = subsystem_results["yard_management"]["strategies"]
ax = axes[0, 1]
ax.bar(
    [name.replace("_", " ").title() for name in yard_strategies],
    [value * 100 for value in yard_strategies.values()],
    color="#59A14F",
    edgecolor="black",
)
ax.set_ylabel("Modeled contribution (percentage points)")
ax.set_title("Yard Capacity Strategy Contributions")
ax.tick_params(axis="x", rotation=30)
ax.grid(True, alpha=0.3, axis="y")

gate_result = subsystem_results["gate_operations"]
ax = axes[0, 2]
ax.bar(
    ["Processing-time\nimprovement"],
    [gate_result["processing_time_improvement"] * 100],
    color="#F28E2B",
    edgecolor="black",
)
ax.set_ylabel("Improvement (%)")
ax.set_title("Gate Rule Output")
ax.grid(True, alpha=0.3, axis="y")

rail_result = subsystem_results["rail_operations"]
ax = axes[1, 0]
ax.bar(
    ["Current share", "Target share"],
    [
        rail_result["current_capacity_share"] * 100,
        rail_result["target_capacity_share"] * 100,
    ],
    color=["#B07AA1", "#76B7B2"],
    edgecolor="black",
)
ax.set_ylabel("Rail capacity share (%)")
ax.set_title("Rail Capacity Rule")
ax.grid(True, alpha=0.3, axis="y")

equipment_result = subsystem_results["equipment_maintenance"]
ax = axes[1, 1]
ax.bar(
    ["Base", "Rule-based target"],
    [
        equipment_result["base_availability"] * 100,
        equipment_result["optimized_availability"] * 100,
    ],
    color=["#9C755F", "#EDC948"],
    edgecolor="black",
)
ax.set_ylabel("Equipment availability (%)")
ax.set_title("Equipment Availability Rule")
ax.set_ylim(0, 100)
ax.grid(True, alpha=0.3, axis="y")

ax = axes[1, 2]
twin_ids = list(coordinator.subsystem_twins)
communication_matrix = np.zeros((len(twin_ids), len(twin_ids)), dtype=int)
for row, twin_id in enumerate(twin_ids):
    twin = coordinator.subsystem_twins[twin_id]
    connected_ids = {connected_twin.twin_id for connected_twin in twin.connected_twins}
    for column, other_id in enumerate(twin_ids):
        communication_matrix[row, column] = int(other_id in connected_ids)

image = ax.imshow(communication_matrix, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(twin_ids)), labels=twin_ids)
ax.set_yticks(range(len(twin_ids)), labels=twin_ids)
ax.set_xlabel("Receiver twin ID")
ax.set_ylabel("Sender twin ID")
ax.set_title("Established Communication Links")
fig.colorbar(image, ax=ax, ticks=[0, 1], label="Link present (0 = no, 1 = yes)")

fig.tight_layout()
plt.show()

In [ ]:
def analyze_vessel_delay_propagation() -> Tuple[Dict, Dict]:
    """Calculate assumed cross-subsystem effects for a vessel-delay scenario."""
    print("=== WHAT-IF ANALYSIS: VESSEL DELAY PROPAGATION ===")

    # These values define a scenario, rather than estimates fitted to operating data.
    delay_hours = 6
    affected_vessels = 3
    container_repositioning = 180
    truck_appointments_affected = 240

    cascading_effects = {
        "vessel_operations": {
            "primary_delay_hours": delay_hours,
            "berth_reallocation_impact_hours": delay_hours * 0.5,
            "crane_redeployment_cost_dollars": delay_hours * 1000,
        },
        "yard_operations": {
            "containers_repositioned": container_repositioning,
            "yard_congestion_increase_fraction": 0.15,
            "additional_handling_cost_dollars": container_repositioning * 25,
        },
        "gate_operations": {
            "truck_appointments_affected": truck_appointments_affected,
            "queue_increase_minutes": 15,
            "customer_satisfaction_change_fraction": -0.2,
        },
        "rail_operations": {
            "train_slots_lost": 1,
            "rail_capacity_utilization_decrease_fraction": 0.05,
            "alternative_transport_cost_dollars": 5000,
        },
    }

    total_delay_cost = sum(
        cascading_effects[subsystem][cost_key]
        for subsystem, cost_key in [
            ("vessel_operations", "crane_redeployment_cost_dollars"),
            ("yard_operations", "additional_handling_cost_dollars"),
            ("rail_operations", "alternative_transport_cost_dollars"),
        ]
    )

    coordinated_response_benefits = {
        "early_detection_savings_dollars": total_delay_cost * 0.3,
        "proactive_repositioning_efficiency_fraction": 0.4,
        "customer_communication_improvement_fraction": 0.5,
        "system_resilience_improvement_fraction": 0.25,
    }

    print(f"\nAssumed vessel delay: {delay_hours} hours")
    print("\nCascading effects:")
    for subsystem, effects in cascading_effects.items():
        print(f"  {subsystem.replace('_', ' ').title()}:")
        for effect, value in effects.items():
            label = effect.replace("_", " ").title()
            if effect.endswith("_dollars"):
                print(f"    - {label}: ${value:,.2f}")
            elif effect.endswith("_fraction"):
                print(f"    - {label}: {value:.1%}")
            else:
                print(f"    - {label}: {value}")

    print("\nSystem-wide impact under the scenario assumptions:")
    print(f"  Sum of specified cost components: ${total_delay_cost:,.2f}")
    print(f"  Affected vessels: {affected_vessels}")
    print(f"  Containers to reposition: {container_repositioning}")
    print(f"  Truck appointments affected: {truck_appointments_affected}")

    print("\nIllustrative coordinated-response assumptions:")
    for benefit, value in coordinated_response_benefits.items():
        label = benefit.replace("_", " ").title()
        if benefit.endswith("_dollars"):
            print(f"  - {label}: ${value:,.2f}")
        else:
            print(f"  - {label}: {value:.1%}")

    return cascading_effects, coordinated_response_benefits


delay_effects, response_benefits = analyze_vessel_delay_propagation()

assert delay_effects["vessel_operations"]["primary_delay_hours"] == 6
assert response_benefits["early_detection_savings_dollars"] >= 0

## Interpreting the example

### What the scenario demonstrates
The coordinator executes five subsystem rules in a visible order, establishes links among the twins, and delivers three status updates to Vessel Operations. The message handlers retain those updates for inspection. The numerical subsystem rules are still evaluated independently: the messages do not feed back into the forecasts or trigger a second optimization pass. This distinction matters because communication infrastructure and operational coordination are related but not identical capabilities.

The plotted values are direct outputs of the scenario rules. For example, the vessel plot separates the base processing-time assumption from the queue and equipment contributions. The communication matrix shows which links are established; it does not measure message volume, latency, or communication quality. The what-if calculation sums only its specified monetary cost components. Its percentage benefits are assumptions, not measured effects.

### Strengths and limitations
**Strengths**
- The subsystem boundaries and coordinator responsibilities are explicit.
- Rule inputs, outputs, units, and scenario assumptions can be inspected in code.
- The message queue illustrates prioritized status delivery, and state history has a bounded memory footprint.
- The separate delay scenario makes cross-subsystem effects and their assumed cost components visible.

**Limitations**
- The rules are illustrative and are not calibrated against terminal data.
- The scenario does not represent time-step simulation, uncertainty, capacity feasibility, or operational scheduling constraints.
- Message exchange does not create a feedback loop or prove that coordination improves performance.
- Cost units in the yard rule and several what-if assumptions are not a complete business case.
- No system-wide optimum, realized KPI improvement, or causal benefit is calculated.

### When this modeling level is useful
A rule-based integrated prototype is useful for explaining subsystem interfaces, organizing assumptions, and identifying which data a more detailed study would require. It should not be used alone to commit resources or promise operational gains. A practical next step is to replace illustrative parameters with documented terminal data and to specify validation criteria, while retaining clear subsystem interfaces and traceable scenario assumptions.